In [ ]:
import time
import numpy as np
import json
import pickle
import websocket
import socket
import os

# --- CONFIGURARE ---
BASE_DEAP_PATH = r"C:\Users\PC\Desktop\Facultate\EEG_BILSTM\kaggle\input\deap-dataset\data_preprocessed_python"
SERVER_URI = "ws://localhost:65432"
CHANNELS_IDX = [0, 2, 3, 6, 7, 10, 11, 13, 16, 19, 20, 24, 25, 28, 29, 31]

def load_data(subject_id):
    file_path = os.path.join(BASE_DEAP_PATH, f"s{subject_id}.dat")
    print(f"📂 Încărcare date pentru S{subject_id}...")
    try:
        with open(file_path, 'rb') as f:
            content = pickle.load(f, encoding='latin1')
        return content['data'], content['labels']
    except Exception as e:
        print(f"❌ Eroare la citire fișier: {e}")
        return None, None

def get_latest_command(ws, blocking=False):
    """ Citește mesajele din socket. Acum știe să asculte Play, Pause și Resume. """
    cmd = None
    if blocking:
        ws.sock.settimeout(None) 
    else:
        ws.sock.settimeout(0.001) 
    
    try:
        while True:
            msg = ws.recv()
            try:
                data = json.loads(msg)
                if isinstance(data, dict) and data.get("type") in ["cmd_start_stream", "cmd_pause_stream", "cmd_resume_stream"]:
                    cmd = data
                    if blocking: return cmd 
            except: pass
    except (socket.timeout, websocket.WebSocketTimeoutException):
        pass
    except Exception as e:
        raise e 
    
    return cmd

def stream_deap():
    ws = websocket.WebSocket()
    headers = {"User-Agent": "Mozilla/5.0 (Compatible)"}
    
    current_subject = None
    data = None
    labels = None

    try:
        print(f"🔌 Conectare la {SERVER_URI}...")
        ws.connect(SERVER_URI, header=headers, timeout=20)
        print("🟢 CONECTAT! Motorul DEAP a pornit.\n")

        state = "WAITING"
        active_cmd = None

        while True:
            # ----------------------------------------------------
            # STAREA 1: Așteptăm comandă de Play de la UI
            # ----------------------------------------------------
            if state == "WAITING":
                print("⏳ Aștept comandă 'Play' de la Interfața Web...")
                active_cmd = get_latest_command(ws, blocking=True)
                if active_cmd["type"] == "cmd_start_stream":
                    state = "PLAYING"

            # ----------------------------------------------------
            # STAREA 2: Rulăm Trial-ul cerut
            # ----------------------------------------------------
            if state == "PLAYING":
                subj = active_cmd["subject"]
                trial_idx = active_cmd["trial"]
                
                if subj != current_subject or data is None:
                    data, labels = load_data(subj)
                    current_subject = subj
                    
                if data is None:
                    state = "WAITING"
                    continue
                
                val_true = labels[trial_idx][0]
                aro_true = labels[trial_idx][1]
                
                print(f"\n▶️ START STREAM -> S{current_subject} | TRIAL {trial_idx + 1}/40")
                print(f"   Target: Valență={val_true:.2f}, Arousal={aro_true:.2f}")

                ws.send(json.dumps({
                    "type": "stream_info",
                    "true_valence": float(val_true),
                    "true_arousal": float(aro_true)
                }))

                trial_signal = data[trial_idx][CHANNELS_IDX, :]
                total_samples = trial_signal.shape[1]
                cursor = 0
                interrupted = False

                while cursor < total_samples:
                    # 1. Verificăm rapid comenzile UI (Pause, Replay)
                    new_cmd = get_latest_command(ws, blocking=False)
                    if new_cmd:
                        if new_cmd["type"] == "cmd_start_stream":
                            print("\n⚠️ Oprit forțat. O nouă comandă Play/Replay a fost primită!")
                            active_cmd = new_cmd
                            interrupted = True
                            break
                        elif new_cmd["type"] == "cmd_pause_stream":
                            print("\n⏸️ Pauză... Aștept Resume.")
                            # Buclă de blocare infinită până vine Resume
                            while True:
                                resume_cmd = get_latest_command(ws, blocking=True)
                                if resume_cmd["type"] == "cmd_resume_stream":
                                    print("▶️ Reluare stream!")
                                    break
                                elif resume_cmd["type"] == "cmd_start_stream":
                                    # Userul a dat Replay în timp ce era pe pauză
                                    active_cmd = resume_cmd
                                    interrupted = True
                                    break
                            if interrupted: break

                    # 2. Trimitem porțiunea de semnal (Chunk)
                    chunk = trial_signal[:, cursor:cursor+16]
                    ws.send(json.dumps(chunk.tolist()))
                    
                    # 3. Trimitem progresul
                    pct = int((cursor / total_samples) * 100)
                    ws.send(json.dumps({"type": "progress", "percent": pct}))

                    cursor += 16
                    time.sleep(0.125) 
                    
                if not interrupted:
                    # Trial-ul a ajuns la 100% natural. 
                    ws.send(json.dumps({"type": "stream_end"}))
                    print("⏹️ Trial terminat cu succes.")
                    state = "WAITING"

    except KeyboardInterrupt:
        print("\nOprire manuală simulator.")
    finally:
        ws.close()

if __name__ == "__main__":
    stream_deap()

🔌 Conectare la ws://localhost:65432...
🟢 CONECTAT! Motorul DEAP a pornit.

⏳ Aștept comandă 'Play' de la Interfața Web...
📂 Încărcare date pentru S01...

▶️ START STREAM -> S01 | TRIAL 1/40
   Target: Valență=7.71, Arousal=7.60

⚠️ Oprit forțat. O nouă comandă Play/Replay a fost primită!

▶️ START STREAM -> S01 | TRIAL 2/40
   Target: Valență=8.10, Arousal=7.31

⚠️ Oprit forțat. O nouă comandă Play/Replay a fost primită!

▶️ START STREAM -> S01 | TRIAL 3/40
   Target: Valență=8.58, Arousal=7.54

⚠️ Oprit forțat. O nouă comandă Play/Replay a fost primită!

▶️ START STREAM -> S01 | TRIAL 4/40
   Target: Valență=4.94, Arousal=6.01

⚠️ Oprit forțat. O nouă comandă Play/Replay a fost primită!

▶️ START STREAM -> S01 | TRIAL 15/40
   Target: Valență=3.17, Arousal=8.08

⚠️ Oprit forțat. O nouă comandă Play/Replay a fost primită!
📂 Încărcare date pentru S10...

▶️ START STREAM -> S10 | TRIAL 1/40
   Target: Valență=7.91, Arousal=6.15
